In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

Classic Bengio style MLP

In [10]:
class MLP(nn.Module):
    def __init__(self, vocab_size,context_length,embedding_dim,hidden_dim):
        super().__init__()
        self.h_in = embedding_dim*context_length
        self.emb_layer = nn.Embedding(vocab_size,embedding_dim)
        self.h = nn.Linear(embedding_dim*context_length,hidden_dim)
        self.bn = nn.BatchNorm1d(hidden_dim)
        self.tanh = nn.Tanh()
        self.out = nn.Linear(hidden_dim,vocab_size)

        

    def forward(self,x):
        x = self.emb_layer(x).reshape(-1,self.h_in)
        x = self.h(x)
        x = self.bn(x)
        x = self.tanh(x)
        o = self.out(x)
        return o


My variant with position embedding

In [15]:
class MLP_PE(nn.Module):
    def __init__(self,vocab_size,context_length,word_embedding_dim,hidden_dim,max_pos,pos_embedding_dim):
        """
        (B,C) → (B, C*E + P) → (B, H) → (B, V)
        """
        super().__init__()
        self.max_pos = max_pos
        self.w_in =  word_embedding_dim*context_length
        self.p_in = pos_embedding_dim
        self.h_in =self.w_in + self.p_in
        self.word_emb_layer = nn.Embedding(vocab_size,word_embedding_dim)
        self.pos_emb_layer = nn.Embedding(max_pos,pos_embedding_dim)
        self.h = nn.Linear(self.h_in,hidden_dim)
        self.bn = nn.BatchNorm1d(hidden_dim)
        self.tanh = nn.Tanh()
        self.out = nn.Linear(hidden_dim,vocab_size)

    def forward(self,x,pos):
        assert pos.shape[0] == x.shape[0]
        pos = torch.clamp(pos,max=self.max_pos -1)
        x_w = self.word_emb_layer(x).reshape(-1,self.w_in) # -> (B,C*E)
        x_p = self.pos_emb_layer(pos) # -> (B,P)
        x = torch.cat((x_w,x_p),dim=1) # -> (B, C*E + P)
        x = self.h(x)
        x = self.bn(x)
        x = self.tanh(x)
        return self.out(x)


In [ ]:
MLP_model = MLP(27,4,10,200)

In [28]:
PosMLP = MLP_PE(27,4,10,172,15,10)

In [13]:
sum(p.numel() for p in MLP_model.parameters())

14297

In [29]:
sum(p.numel() for p in PosMLP.parameters())

14207

In [ ]:
from torch.utils.data import Dataset
from torch.utils.data import DataLoader


class NamesDataset(Dataset):
    def __init__(self,names:list,stoi:dict,context_length:int):
        super().__init__()
        Xs = []
        ys = []
        pos = []
        for name in names:
            chars = "."*context_length + name + "."
            p = 0
            for i in range(len(chars) - context_length):
                x = [stoi[chars[j]] for j in range(i,i+context_length)]
                y = stoi[chars[i+context_length]]
                pos.append(p)
                p+=1

                Xs.append(x)
                ys.append(y)

        self.x = torch.tensor(Xs)
        self.y = torch.tensor(ys)
        self.pos = torch.tensor(pos)


    def __getitem__(self, index):
        return self.x[index],self.pos[index],self.y[index]
    
    def __len__(self):
        return len(self.y)
    



Setup Data for experiments across hyperparameter search space

In [ ]:
CONFIG = {
    'context_length':4,
    'batch_size' : 512
}

In [ ]:
import random

with open("data/names.txt","r") as f:
    names = f.read().splitlines()

random.seed(123)
random.shuffle(names)

tr_idx = int(0.8 * len(names))
val_idx = int(0.9 * len(names))

vocab = list(set(''.join(names)))
vocab.sort()
stoi = {k:v+1 for v,k in enumerate(vocab)}
stoi['.'] = 0

train_dataset = NamesDataset(names[:tr_idx],stoi,CONFIG['context_length'])
val_dataset = NamesDataset(names[tr_idx:val_idx],stoi,CONFIG['context_length'])
test_dataset = NamesDataset(names[val_idx:],stoi,CONFIG['context_legth'])

train_dataloader = DataLoader(dataset=train_dataset,
                              batch_size=CONFIG['batch_size'],
                              shuffle=True,
                              num_workers=0)

val_dataloader = DataLoader(dataset=val_dataset,
                              batch_size=CONFIG['batch_size'],
                              shuffle=False,
                              num_workers=0)

test_dataloader = DataLoader(dataset=test_dataset,
                              batch_size=CONFIG['batch_size'],
                              shuffle=False,
                              num_workers=0)